# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/barak-hub/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**My Baseline Scoring Rule**

I will prioritize webpages that receive many Google Search impressions but have a low click-through rate (CTR).

These webpages may have opportunities to improve their titles and descriptions to attract more clicks.

**Reason codes:**
- `HIGH_IMPRESSIONS_LOW_CTR`: A webpage is frequently shown but receives relatively few clicks.
- `LOW_SEARCH_VOLUME`: A webpage has too few impressions to confidently prioritize.
- `NO_GSC_DATA`: Search performance data is unavailable.
- `NO_ACTION`: No immediate improvement is recommended.

The scoring rule will use historical data only, without future clicks or prediction labels.

In [34]:
from google.colab import userdata
from huggingface_hub import hf_hub_download
import pandas as pd

token = userdata.get("HF_TOKEN")

file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_query_90d.parquet",
    repo_type="dataset",
    token=token
)

df = pd.read_parquet(file_path)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

Dataset shape: (2414248, 21)
Columns: ['client_hash_id', 'content_hash_id', 'query_hash_id', 'query_char_count', 'query_token_count', 'window_start', 'window_end', 'impressions_90d', 'clicks_90d', 'impressions_last30', 'clicks_last30', 'impressions_prev30', 'clicks_prev30', 'avg_position_90d', 'avg_position_last30', 'avg_position_prev30', 'content_total_impressions_90d', 'content_visible_query_count', 'rare_query_count', 'rare_impressions_share', 'anonymized_impressions_share']


In [35]:
for column in df.columns:
    print(column)

client_hash_id
content_hash_id
query_hash_id
query_char_count
query_token_count
window_start
window_end
impressions_90d
clicks_90d
impressions_last30
clicks_last30
impressions_prev30
clicks_prev30
avg_position_90d
avg_position_last30
avg_position_prev30
content_total_impressions_90d
content_visible_query_count
rare_query_count
rare_impressions_share
anonymized_impressions_share


In [36]:
# Calculate CTR before running the signal checks
df["ctr_90d"] = (
    df["clicks_90d"] /
    df["impressions_90d"].replace(0, float("nan"))
)

print("CTR column created successfully!")
print(df[["impressions_90d", "clicks_90d", "ctr_90d"]].head())

CTR column created successfully!
   impressions_90d  clicks_90d  ctr_90d
0               11           0      0.0
1               13           0      0.0
2               16           0      0.0
3               55           0      0.0
4               14           0      0.0


In [37]:
import pandas as pd
import numpy as np

signal1 = df[
    (df["impressions_90d"] > 0) &
    (df["avg_position_90d"].notna())
].copy()

signal1["position_bucket"] = pd.cut(
    signal1["avg_position_90d"],
    bins=[0, 3, 10, 20, np.inf],
    labels=["1-3", "4-10", "11-20", "21+"],
    include_lowest=True
)

signal1_table = (
    signal1.groupby("position_bucket", observed=True)
    .agg(
        n=("ctr_90d", "size"),
        impressions=("impressions_90d", "sum"),
        clicks=("clicks_90d", "sum")
    )
)

signal1_table["ctr"] = (
    signal1_table["clicks"] /
    signal1_table["impressions"]
)

display(signal1_table)

,n,impressions,clicks,ctr
position_bucket,,,,
1-3,405442,31615691,147951,0.004680
4-10,978489,132134544,273863,0.002073
11-20,327380,19669342,25712,0.001307
21+,702937,28323439,12392,0.000438


### Signal Check 1: CTR vs Search Position

**Hypothesis:** Records appearing in better Google Search positions tend to have higher click-through rates.

**Observed results:** Records in positions 1–3 had the highest CTR (0.468%), while records in positions 21+ had the lowest CTR (0.044%).

CTR decreased consistently as search position worsened.

**Verdict: CONFIRMED**

**Interpretation:** The data supports a directional relationship between search ranking position and CTR. However, this is an observational analysis and does not establish causation.

**Decision-support implication:** Low CTR should not automatically trigger a recommendation to optimize titles or descriptions. Search position should also be considered before making an optimization decision.

In [38]:
signal2 = df[df["impressions_90d"] > 0].copy()

signal2["volume_bucket"] = pd.cut(
    signal2["impressions_90d"],
    bins=[0, 10, 100, 1000, float("inf")],
    labels=["1-10", "11-100", "101-1000", "1001+"],
    include_lowest=True
)

signal2_table = (
    signal2.groupby("volume_bucket", observed=True)
    .agg(
        n=("impressions_90d", "size"),
        impressions=("impressions_90d", "sum"),
        clicks=("clicks_90d", "sum")
    )
)

signal2_table["ctr"] = (
    signal2_table["clicks"] /
    signal2_table["impressions"]
)

display(signal2_table)

,n,impressions,clicks,ctr
volume_bucket,,,,
1-10,178402,1784020,3273,0.001835
11-100,1900984,55817314,114918,0.002059
101-1000,311880,80905355,204801,0.002531
1001+,22982,73236327,136926,0.001870


### Signal Check 2: Search Volume vs CTR

**Hypothesis:** Higher search impression volume is associated with higher click-through rates.

**Observed results:** CTR increased from 0.184% in the 1–10 impressions bucket to 0.253% in the 101–1,000 impressions bucket. However, CTR decreased to 0.187% for records with more than 1,000 impressions.

**Verdict: MIXED**

**Interpretation:** The observed relationship between search volume and CTR is not consistently increasing. Higher impressions do not automatically indicate stronger engagement.

**Decision-support implication:** Search volume should be considered alongside CTR and ranking position when prioritizing optimization opportunities. These results are observational and do not establish causation.

In [39]:
df["ctr_90d"] = (
    df["clicks_90d"] /
    df["impressions_90d"].replace(0, float("nan"))
)

print(
    df[["impressions_90d", "clicks_90d", "ctr_90d"]]
    .head(10)
)

   impressions_90d  clicks_90d  ctr_90d
0               11           0      0.0
1               13           0      0.0
2               16           0      0.0
3               55           0      0.0
4               14           0      0.0
5               40           0      0.0
6               16           0      0.0
7               14           0      0.0
8               34           0      0.0
9               11           0      0.0


In [40]:
print("Total records:", len(df))
print("Records with 100+ impressions:",
      (df["impressions_90d"] >= 100).sum())

print("Records with 100+ impressions and CTR below 2%:",
      ((df["impressions_90d"] >= 100) &
       (df["ctr_90d"] < 0.02)).sum())

Total records: 2414248
Records with 100+ impressions: 338105
Records with 100+ impressions and CTR below 2%: 332498


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [41]:
import numpy as np

df["reason_code"] = np.select(
    [
        df["impressions_90d"] < 100,
        (df["impressions_90d"] >= 100) & (df["ctr_90d"] < 0.02)
    ],
    [
        "LOW_SEARCH_VOLUME",
        "HIGH_IMPRESSIONS_LOW_CTR"
    ],
    default="NO_ACTION"
)

df["action_score"] = np.where(
    df["reason_code"] == "HIGH_IMPRESSIONS_LOW_CTR",
    df["impressions_90d"] * (0.02 - df["ctr_90d"]),
    0
)

print(df["reason_code"].value_counts())
print(df["action_score"].describe())

reason_code
LOW_SEARCH_VOLUME           2076143
HIGH_IMPRESSIONS_LOW_CTR     332498
NO_ACTION                      5607
Name: count, dtype: int64
count    2.414248e+06
mean     1.146697e+00
std      1.818512e+01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      1.080588e+04
Name: action_score, dtype: float64


In [42]:
from pathlib import Path

ranked = df.sort_values(
    "action_score",
    ascending=False
).copy()

ranked["action_label"] = np.where(
    ranked["reason_code"] == "HIGH_IMPRESSIONS_LOW_CTR",
    "OPTIMIZE_CTR",
    "MONITOR"
)

output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

ranked.to_csv(output_path, index=False)

print("CSV saved:", output_path)
print("Total ranked records:", len(ranked))
print(ranked[
    ["impressions_90d", "ctr_90d", "action_score",
     "reason_code", "action_label"]
].head(10))

CSV saved: work/outputs/baseline_action_score.csv
Total ranked records: 2414248
         impressions_90d   ctr_90d  action_score               reason_code  \
1713312           543044  0.000101      10805.88  HIGH_IMPRESSIONS_LOW_CTR   
757739            331832  0.000000       6636.64  HIGH_IMPRESSIONS_LOW_CTR   
1832960           292374  0.000000       5847.48  HIGH_IMPRESSIONS_LOW_CTR   
325356            282446  0.000000       5648.92  HIGH_IMPRESSIONS_LOW_CTR   
1968784           278990  0.000000       5579.80  HIGH_IMPRESSIONS_LOW_CTR   
393119            264399  0.000000       5287.98  HIGH_IMPRESSIONS_LOW_CTR   
1925457           256491  0.000008       5127.82  HIGH_IMPRESSIONS_LOW_CTR   
1726168           248191  0.000000       4963.82  HIGH_IMPRESSIONS_LOW_CTR   
1320016           261191  0.002998       4440.82  HIGH_IMPRESSIONS_LOW_CTR   
1600720           209105  0.000000       4182.10  HIGH_IMPRESSIONS_LOW_CTR   

         action_label  
1713312  OPTIMIZE_CTR  
757739   OPTI

## 3. Top-20 review

### Top-20 Review

The top 20 records are ranked by their estimated click improvement opportunity.

All 20 records are classified as `HIGH_IMPRESSIONS_LOW_CTR`, with the recommended action `OPTIMIZE_CTR`.

The highest-ranked record received 543,044 impressions but only 55 clicks, suggesting a possible opportunity to improve its click-through rate.

However, high impressions and low CTR do not automatically mean that a webpage is performing poorly. Search ranking position, search intent, and query relevance could explain the results.

Before recommending changes, I would verify these factors. The ranking is a prioritization tool, not proof that optimization will increase clicks.

In [43]:
review = top20[
    ["impressions_90d", "clicks_90d", "reason_code", "action_label"]
].copy()

review["why_selected"] = (
    "High search visibility but low click-through rate"
)

review["what_could_make_it_wrong"] = (
    "Low search position, irrelevant queries, or search intent "
    "may explain the low CTR"
)

review.index = range(1, 21)

display(review)

,impressions_90d,clicks_90d,reason_code,action_label,why_selected,what_could_make_it_wrong
1,543044,55,HIGH_IMPRESSIONS_LOW_CTR,OPTIMIZE_CTR,High search visibility but low click-through rate,"Low search position, irrelevant queries, or se..."
2,331832,0,HIGH_IMPRESSIONS_LOW_CTR,OPTIMIZE_CTR,High search visibility but low click-through rate,"Low search position, irrelevant queries, or se..."
3,292374,0,HIGH_IMPRESSIONS_LOW_CTR,OPTIMIZE_CTR,High search visibility but low click-through rate,"Low search position, irrelevant queries, or se..."
4,282446,0,HIGH_IMPRESSIONS_LOW_CTR,OPTIMIZE_CTR,High search visibility but low click-through rate,"Low search position, irrelevant queries, or se..."
5,278990,0,HIGH_IMPRESSIONS_LOW_CTR,OPTIMIZE_CTR,High search visibility but low click-through rate,"Low search position, irrelevant queries, or se..."
6,264399,0,HIGH_IMPRESSIONS_LOW_CTR,OPTIMIZE_CTR,High search visibility but low click-through rate,"Low search position, irrelevant queries, or se..."
7,256491,2,HIGH_IMPRESSIONS_LOW_CTR,OPTIMIZE_CTR,High search visibility but low click-through rate,"Low search position, irrelevant queries, or se..."
8,248191,0,HIGH_IMPRESSIONS_LOW_CTR,OPTIMIZE_CTR,High search visibility but low click-through rate,"Low search position, irrelevant queries, or se..."
9,261191,783,HIGH_IMPRESSIONS_LOW_CTR,OPTIMIZE_CTR,High search visibility but low click-through rate,"Low search position, irrelevant queries, or se..."
10,209105,0,HIGH_IMPRESSIONS_LOW_CTR,OPTIMIZE_CTR,High search visibility but low click-through rate,"Low search position, irrelevant queries, or se..."


In [44]:
review["what_could_make_it_wrong"] = review.apply(
    lambda row:
    "Zero clicks may reflect poor ranking or irrelevant searches"
    if row["clicks_90d"] == 0
    else "Some clicks exist; check ranking position and search intent before changing the page",
    axis=1
)

review["why_selected"] = review.apply(
    lambda row:
    f"{int(row['impressions_90d']):,} impressions but only "
    f"{int(row['clicks_90d']):,} clicks",
    axis=1
)

display(review.to_string())

'    impressions_90d  clicks_90d               reason_code  action_label                             why_selected                                                              what_could_make_it_wrong\n1            543044          55  HIGH_IMPRESSIONS_LOW_CTR  OPTIMIZE_CTR   543,044 impressions but only 55 clicks  Some clicks exist; check ranking position and search intent before changing the page\n2            331832           0  HIGH_IMPRESSIONS_LOW_CTR  OPTIMIZE_CTR    331,832 impressions but only 0 clicks                           Zero clicks may reflect poor ranking or irrelevant searches\n3            292374           0  HIGH_IMPRESSIONS_LOW_CTR  OPTIMIZE_CTR    292,374 impressions but only 0 clicks                           Zero clicks may reflect poor ranking or irrelevant searches\n4            282446           0  HIGH_IMPRESSIONS_LOW_CTR  OPTIMIZE_CTR    282,446 impressions but only 0 clicks                           Zero clicks may reflect poor ranking or irrelevant searches\

In [45]:
print("Total reviews:", len(review))
print("Missing explanations:",
      review[["why_selected", "what_could_make_it_wrong"]]
      .isna().sum().sum())

Total reviews: 20
Missing explanations: 0


In [46]:
top20 = ranked.head(20).copy()

display(top20[
    [
        "client_hash_id",
        "content_hash_id",
        "impressions_90d",
        "clicks_90d",
        "ctr_90d",
        "action_score",
        "reason_code",
        "action_label"
    ]
])

,client_hash_id,content_hash_id,impressions_90d,clicks_90d,ctr_90d,action_score,reason_code,action_label
1713312,client_8ddc46da5414ffd8,content_943dc881428182b8,543044,55,0.000101,10805.88,HIGH_IMPRESSIONS_LOW_CTR,OPTIMIZE_CTR
757739,client_0fa64a184f18a4a0,content_11bf4c33adea7bdc,331832,0,0.000000,6636.64,HIGH_IMPRESSIONS_LOW_CTR,OPTIMIZE_CTR
1832960,client_8ddc46da5414ffd8,content_d0acf7062bc6b257,292374,0,0.000000,5847.48,HIGH_IMPRESSIONS_LOW_CTR,OPTIMIZE_CTR
325356,client_23a62021009f63c4,content_c60628276389acbb,282446,0,0.000000,5648.92,HIGH_IMPRESSIONS_LOW_CTR,OPTIMIZE_CTR
1968784,client_cd12bcfd98942aa1,content_99fc6465edb0e52c,278990,0,0.000000,5579.80,HIGH_IMPRESSIONS_LOW_CTR,OPTIMIZE_CTR
393119,client_1a730cb2640a1abf,content_39e19a3ec2d95f9d,264399,0,0.000000,5287.98,HIGH_IMPRESSIONS_LOW_CTR,OPTIMIZE_CTR
1925457,client_8ddc46da5414ffd8,content_32c5cc913fb4ff41,256491,2,0.000008,5127.82,HIGH_IMPRESSIONS_LOW_CTR,OPTIMIZE_CTR
1726168,client_a80fca3f171ed1de,content_012de75c008aa653,248191,0,0.000000,4963.82,HIGH_IMPRESSIONS_LOW_CTR,OPTIMIZE_CTR
1320016,client_73cda7b4e4f265ea,content_987d251ee617d9c6,261191,783,0.002998,4440.82,HIGH_IMPRESSIONS_LOW_CTR,OPTIMIZE_CTR
1600720,client_8ddc46da5414ffd8,content_7471467133493ce6,209105,0,0.000000,4182.10,HIGH_IMPRESSIONS_LOW_CTR,OPTIMIZE_CTR


## 4. Weak picks + leakage check

### Weak Picks Review

I examined 10 records with at least 100 impressions and a CTR of 2% or higher.

All 10 received the `NO_ACTION` reason code and an action score of zero.

These records demonstrate why high search visibility alone is not enough to recommend optimization.

However, a CTR above 2% does not guarantee good performance. Search position, user intent, and business goals may still justify further investigation.

The 2% threshold is a baseline assumption, not a universal SEO standard.

In [47]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [48]:
weak_picks = ranked[
    (ranked["impressions_90d"] >= 100) &
    (ranked["ctr_90d"] >= 0.02)
].head(10)

print("Records with sufficient impressions and CTR >= 2%:")
print(len(weak_picks))

display(weak_picks[
    ["impressions_90d", "clicks_90d",
     "ctr_90d", "reason_code", "action_score"]
])

Records with sufficient impressions and CTR >= 2%:
10


,impressions_90d,clicks_90d,ctr_90d,reason_code,action_score
1654520,146,4,0.027397,NO_ACTION,0.0
1660811,215,5,0.023256,NO_ACTION,0.0
1660193,325,8,0.024615,NO_ACTION,0.0
1660341,150,3,0.020000,NO_ACTION,0.0
1659886,126,3,0.023810,NO_ACTION,0.0
1662847,178,12,0.067416,NO_ACTION,0.0
1317211,262,7,0.026718,NO_ACTION,0.0
1661602,100,2,0.020000,NO_ACTION,0.0
1656164,194,4,0.020619,NO_ACTION,0.0
1655550,137,4,0.029197,NO_ACTION,0.0


In [49]:
from pathlib import Path
import pandas as pd

csv_path = Path("work/outputs/baseline_action_score.csv")

print("CSV exists:", csv_path.exists())

if csv_path.exists():
    check = pd.read_csv(
        csv_path,
        usecols=[
            "action_score",
            "reason_code",
            "action_label"
        ]
    )

    print("Total records:", len(check))
    print("Required columns:", check.columns.tolist())
    print("Missing action labels:",
          check["action_label"].isna().sum())

    print("Scores ranked correctly:",
          check["action_score"].is_monotonic_decreasing)

CSV exists: True
Total records: 2414248
Required columns: ['reason_code', 'action_score', 'action_label']
Missing action labels: 0
Scores ranked correctly: True


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.